# LSTM_VAL: lookback selected by validation loss

For every stock and re-estimation window, this notebook picks the fixed-lookback LSTM (L in {10, 21, 42, 63}) whose ensemble forecast has the lowest **QLIKE on the validation year**, and uses that model's forecasts for the following test window. This is the conventional, data-driven alternative to AIC: tuning the lookback like any other hyperparameter, at the cost of training the whole grid.

No extra training is needed. Each fixed-lookback run in `NN/NN.ipynb` writes `Results/LSTM_<L>/LSTM_<L>_windows.csv` with the validation loss of every stock-window. Because every lookback uses the same training, validation and test dates, the validation losses are directly comparable across runs. Ties are broken in favour of the shorter lookback.

Output: `Results/LSTM_VAL/` in the same layout as the other models, so the DM, RU and MCS notebooks pick it up automatically. Set `MODEL_SUFFIX = "_refit12m"` to build the yearly-refit version from the yearly-refit grid.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

GRID = [10, 21, 42, 63]
MODEL_SUFFIX = ""                  # "" for the main (annual-refit) runs, "_refit1m" for monthly refits
CRITERION = "val_qlike"            # selection criterion recorded in the window logs
OUTPUT_MODEL = f"LSTM_VAL{MODEL_SUFFIX}"

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Results").is_dir() and (candidate / "Data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a project root containing Results/ and Data/.")


ROOT = find_project_root(Path.cwd())
RESULTS_DIR = ROOT / "Results"
grid_models = [f"LSTM_{L}{MODEL_SUFFIX}" for L in GRID]

window_logs = []
for model in grid_models:
    path = RESULTS_DIR / model / f"{model}_windows.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Re-run NN/NN.ipynb for {model} with the current code, "
            "which writes the per-window validation losses."
        )
    log = pd.read_csv(path)
    log["model"] = model
    window_logs.append(log)
windows = pd.concat(window_logs, ignore_index=True)

# Every grid model must cover the same stock-windows with the same dates.
keys = ["ticker", "test_start", "test_end_exclusive"]
coverage = windows.groupby(keys)["model"].nunique()
incomplete = coverage[coverage != len(grid_models)]
if not incomplete.empty:
    raise ValueError(f"{len(incomplete)} stock-windows are missing from some grid models:\n{incomplete.head()}")
if not windows.groupby(["model"])["lookback"].nunique().eq(1).all():
    raise ValueError("A fixed-lookback window log contains more than one lookback.")

print(f"Loaded {len(windows):,} window records from {', '.join(grid_models)}")

In [ ]:
# Lowest validation loss per stock-window; ties go to the shorter lookback.
ranked = windows.sort_values(keys + [CRITERION, "lookback"])
selection = ranked.groupby(keys, as_index=False).first()[keys + ["model", "lookback", CRITERION]]
selection = selection.rename(columns={"model": "selected_model", "lookback": "selected_lookback"})

shares = (selection["selected_lookback"].value_counts(normalize=True)
          .reindex(GRID, fill_value=0.0).rename("share_of_stock_windows"))
display(shares.to_frame().style.format("{:.1%}"))
display(selection.groupby("ticker")["selected_lookback"].agg(["mean", "median", "min", "max"]))

In [ ]:
# Assemble the selected forecasts from each grid model's individual files.
individual = {}
for model in grid_models:
    frames = []
    for path in sorted((RESULTS_DIR / model / "individual").glob("*.csv")):
        frame = pd.read_csv(path, parse_dates=["date"])
        frames.append(frame)
    individual[model] = pd.concat(frames, ignore_index=True)

pieces = []
for row in selection.itertuples(index=False):
    source = individual[row.selected_model]
    start, end = pd.Timestamp(row.test_start), pd.Timestamp(row.test_end_exclusive)
    piece = source.loc[(source["ticker"] == row.ticker)
                       & (source["date"] >= start) & (source["date"] < end)].copy()
    piece["lookback_period"] = int(row.selected_lookback)
    pieces.append(piece)
detail = pd.concat(pieces, ignore_index=True).sort_values(["ticker", "date"])
detail = detail[["ticker", "date", "real", "predicted", "mse", "qlike", "lookback_period"]]
if detail.duplicated(["ticker", "date"]).any():
    raise ValueError("Duplicate stock-dates in the assembled LSTM_VAL forecasts.")

# Same global calendar and column order as the grid models' wide files.
template = pd.read_csv(RESULTS_DIR / grid_models[0] / f"{grid_models[0]}.csv", parse_dates=["date"]).set_index("date")
wide = detail.pivot(index="date", columns="ticker", values="predicted").reindex(
    index=template.index, columns=template.columns
)
print(f"Assembled {len(detail):,} forecasts; grid model has {int(template.notna().sum().sum()):,}")

In [ ]:
output_dir = RESULTS_DIR / OUTPUT_MODEL
individual_dir = output_dir / "individual"
individual_dir.mkdir(parents=True, exist_ok=True)

for ticker, frame in detail.groupby("ticker"):
    frame.assign(date=frame["date"].dt.strftime("%Y-%m-%d")).to_csv(individual_dir / f"{ticker}.csv", index=False)
wide.to_csv(output_dir / f"{OUTPUT_MODEL}.csv", index_label="date")
selection.to_csv(output_dir / f"{OUTPUT_MODEL}_selection.csv", index=False)

# Equal-weight cross-stock evaluation, as in the model notebooks.
per_stock = detail.groupby("ticker").agg(mse=("mse", "mean"), qlike=("qlike", "mean"))
evaluation = pd.DataFrame([{
    "model": OUTPUT_MODEL,
    "requested_tickers": int(wide.shape[1]),
    "evaluated_tickers": int(len(per_stock)),
    "observations": int(len(detail)),
    "mse": float(per_stock["mse"].mean()),
    "qlike": float(per_stock["qlike"].mean()),
}])
evaluation.to_csv(output_dir / f"{OUTPUT_MODEL}_evaluation.csv", index=False)

config = {
    "model": OUTPUT_MODEL,
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "grid_models": grid_models,
    "selection_criterion": CRITERION,
    "tie_break": "shorter lookback",
    "selection_shares": {str(k): float(v) for k, v in shares.items()},
    "stock_windows": int(len(selection)),
    "training_cost_note": "uses the trained networks of every grid model; see their *_windows.csv train_seconds",
}
with (output_dir / f"{OUTPUT_MODEL}_config.json").open("w", encoding="utf-8") as handle:
    json.dump(config, handle, indent=2)
display(evaluation)
print(f"Saved {OUTPUT_MODEL} to {output_dir}")